# 12b — View Cutouts for a Gaia-stable diaObject

## Purpose

Visualise the triplet of image stamps **(Science / Template / Difference / Science−Template)**
for every diaSource detection of a user-selected `diaObjectId`, in order to demonstrate
that **dipole residuals** in the PSF-subtracted difference image are responsible for
triggering spurious Rubin AP alerts on intrinsically stable Gaia stars.

## Layout — 2×3 compact grid per diaSource (presentation-ready)

| Position | Content |
|----------|---------|
| (0,0) | **Info panel** — object metadata, fluxes, magnitudes, dipole flag, dates |
| (0,1) | **Science** image — shared vmin/vmax with Template |
| (0,2) | **Template** image — shared vmin/vmax with Science |
| (1,0) | **Light curve** in the current band (src+fp, current visit highlighted) |
| (1,1) | **DIA Difference** (downloaded) — symmetric ±vmax, diverging colormap |
| (1,2) | **Sci − Template** (computed) — same ±vmax, diverging colormap |

## Data sources

| Data | Location |
|------|----------|
| Cutout `.npy` arrays + all metadata (incl. dipole flags) | `fullcutouts_{diaObjectId}/manifest.csv` |
| Forced photometry | `fullcutouts_{diaObjectId}/manifest_fp.csv` |
| Gaia DR3 object-level metadata | `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_matched.csv` |

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-13
- **Last update:** 2026-05-14 — compact 2×3 layout
- **Subject:** Fink/LSST DIA — Dipole hypothesis — cutout visualisation


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
%matplotlib inline

In [ ]:
# try:
#    import ipympl
#    %matplotlib widget
#    print("ipympl → interactive backend")
# except ImportError:
#    %matplotlib inline
#    print("inline backend")

| diaObjectId        |
| ------------------ |
| 170032917163016282 |
| 170094456334188639 |
| 170019717277810735 |
| 170235219334922248 |
| 170094456400773135 |
| 170226393328648251 |
| 170257199649521669 |
| 170094456357257301 |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────

# diaObjectId to inspect — must have a fullcutouts_{id}/ directory

objsid = {
    0: 170032917163016282,
    1: 170094456334188639,
    2: 170019717277810735,
    3: 170235219334922248,
    4: 170094456400773135,
    5: 170226393328648251,
    6: 170257199649521669,
    7: 170094456357257301,
}

DIAOBJECT_ID = objsid[3]

# Bands to display: None = all, or e.g. ["r", "i"]
BANDS_FILTER = None

# Max rows per band (None = unlimited)
MAX_ROWS_PER_BAND = None

# Image stretch for science/template panels: 'zscale' (percentile 1–99)
STRETCH_MODE = "zscale"

# Color maps
CMAP_SCI = "gray"
CMAP_DIFF = "RdBu_r"  # diverging: red = positive, blue = negative

# Save figures to DIR_FIGS?
SAVE_FIGS = True
DIR_FIGS = "figs_FINK_BLOCK_LC_12b"

# ─────────────────────────────────────────────────────────────────────────────
# Fixed paths
# ─────────────────────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest.csv")
FILE_MANIFEST_FP = os.path.join(DIR_CUTOUTS, "manifest_fp.csv")
FILE_GAIA_MATCHED = os.path.join("data_FINK_BLOCK_LC_08b", "fink_diaobj_gaia_join_matched.csv")

AB_FLUX_ZERO = 3631e9  # nJy → m_AB = −2.5 × log10(f / AB_FLUX_ZERO)
BAND_ORDER = list("ugrizy")
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}

os.makedirs(DIR_FIGS, exist_ok=True)
print(f"diaObjectId : {DIAOBJECT_ID}")
print(f"Manifest    : {os.path.abspath(FILE_MANIFEST)}")
print(f"Manifest fp : {os.path.abspath(FILE_MANIFEST_FP)}")
print(f"Figures     : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def flux_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive flux."""
    try:
        f = float(flux_nJy)
    except (TypeError, ValueError):
        return np.nan
    return -2.5 * np.log10(f / AB_FLUX_ZERO) if (np.isfinite(f) and f > 0) else np.nan


def flux_to_luptitude(flux_nJy, b_soft=None):
    """Convert flux (nJy) to asinh magnitude (Luptitude, Lupton 1999).
    If b_soft is None it is set to median(|flux|)/3 as a fallback.
    """
    f = np.asarray(flux_nJy, dtype=float)
    if b_soft is None or not np.isfinite(b_soft) or b_soft <= 0:
        b_soft = max(float(np.nanmedian(np.abs(f))) / 3.0, 1.0)
    log10_e_inv = 1.085736
    lup = -2.5 * log10_e_inv * (np.arcsinh(f / (2.0 * b_soft)) + np.log(b_soft)) + (
        31.4 + 2.5 * np.log10(b_soft)
    )
    return lup, b_soft


def zscale(arr, lo=1.0, hi=99.0):
    """Return (vmin, vmax) from percentile-based z-scale."""
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return 0.0, 1.0
    return float(np.percentile(finite, lo)), float(np.percentile(finite, hi))


def symvlim(arr, percentile=99.5):
    """Return the symmetric half-range vmax for a diverging image display."""
    finite = arr[np.isfinite(arr)]
    v = float(np.percentile(np.abs(finite), percentile)) if len(finite) else 1.0
    return max(v, 1e-9)


def load_npy(src_id, band, kind):
    """Load fullcutouts_{obj}/cutouts/{src_id}_{band}_{kind}.npy. Returns float32 or None."""
    fpath = os.path.join(DIR_CUTOUTS, "cutouts", f"{src_id}_{band}_{kind}.npy")
    return np.load(fpath).astype(np.float32) if os.path.exists(fpath) else None


def parse_bool(val):
    """Coerce a dipole flag (bool / int / str / NaN) to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def mjd_to_date(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


def draw_crosshair(ax, arr):
    """Draw a centred crosshair on an image axis."""
    if arr is None:
        return
    ny, nx = arr.shape
    ax.axvline(nx / 2, color="yellow", lw=0.9, ls="--", alpha=0.8)
    ax.axhline(ny / 2, color="yellow", lw=0.9, ls="--", alpha=0.8)


print("Utility functions defined.")

## 3. Load manifest (flux + dipole flags)

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(
        f"{FILE_MANIFEST} not found.\nRun: python fink_download_full_cutouts.py --obj_id {DIAOBJECT_ID}"
    )

df = pd.read_csv(FILE_MANIFEST)
df["isDipole"] = df["r:isDipole"].fillna(False).apply(parse_bool)
df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)

n_dip = df["isDipole"].sum()
print(f"Manifest : {len(df)} diaSources  |  {n_dip} dipoles ({100 * n_dip / len(df):.1f}%)")
print(f"Bands    : {sorted(df['r:band'].unique())}")
df[
    [
        "r:diaSourceId",
        "r:band",
        "r:visit",
        "r:detector",
        "r:psfFlux",
        "r:scienceFlux",
        "r:templateFlux",
        "isDipole",
    ]
].head()

## 4. Load Gaia DR3 object-level metadata

In [ ]:
gaia_G_mag, gaia_status, fink_group, gaia_dr3_id = np.nan, "?", "?", "?"
gaia_ra, gaia_dec = np.nan, np.nan

if os.path.exists(FILE_GAIA_MATCHED):
    df_gaia = pd.read_csv(FILE_GAIA_MATCHED)
    hit = df_gaia[df_gaia["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        gaia_G_mag = float(r.get("gaia_phot_g_mean_mag", np.nan))
        gaia_status = str(r.get("gaia_status", "?"))
        fink_group = str(r.get("group", "?"))
        gaia_dr3_id = str(r.get("dr3_source_id", "?"))
        gaia_ra = float(r.get("ra", r.get("gaia_ra", np.nan)))
        gaia_dec = float(r.get("dec", r.get("gaia_dec", np.nan)))

G_str = f"{gaia_G_mag:.2f}" if np.isfinite(gaia_G_mag) else "?"
print(f"Gaia G={G_str}  |  status={gaia_status}  |  group={fink_group}  |  DR3={gaia_dr3_id}")

## 5. Load forced photometry

In [ ]:
if os.path.exists(FILE_MANIFEST_FP):
    df_fp = pd.read_csv(FILE_MANIFEST_FP).sort_values("r:midpointMjdTai").reset_index(drop=True)
    print(f"Forced photometry : {len(df_fp)} points  bands: {sorted(df_fp['r:band'].unique())}")
else:
    df_fp = pd.DataFrame()
    print(f"No manifest_fp.csv found in {DIR_CUTOUTS}/ — fp not shown.")

# Common time origin: earliest point across src + fp
t0_cands = [df["r:midpointMjdTai"].min()]
if not df_fp.empty:
    t0_cands.append(df_fp["r:midpointMjdTai"].min())
T0_MJD = min(t0_cands)
T0_DATE = mjd_to_date(T0_MJD)
print(f"Time origin t0 = MJD {T0_MJD:.4f}  ({T0_DATE})")

# Overall first / last detection dates (src only)
DATE_FIRST = mjd_to_date(df["r:midpointMjdTai"].min())
DATE_LAST = mjd_to_date(df["r:midpointMjdTai"].max())
TSPAN_DAYS = df["r:midpointMjdTai"].max() - df["r:midpointMjdTai"].min()
print(f"First detection : {DATE_FIRST}   Last: {DATE_LAST}   Span: {TSPAN_DAYS:.1f} days")

## 6. Compact 2×3 cutout viewer — one figure per diaSource

Layout per diaSource:

```
┌─────────────┬──────────────┬──────────────┐
│  Info panel │   Science    │   Template   │  ← row 0
│   (text)    │  (shared     │  (shared     │
│             │   vmin/vmax) │   vmin/vmax) │
├─────────────┼──────────────┼──────────────┤
│ Light curve │  DIA diff    │  Sci−Tmpl    │  ← row 1
│  (band, pt  │ (downloaded) │  (computed)  │
│  highlighted│  ±vmax sym.  │  ±vmax sym.  │
└─────────────┴──────────────┴──────────────┘
```

In [ ]:
bands_available = [b for b in BAND_ORDER if b in df["r:band"].values]
bands_to_plot = [b for b in bands_available if BANDS_FILTER is None or b in BANDS_FILTER]
print(f"Bands to plot: {bands_to_plot}")


def _show_img_shared(ax, arr, cmap, vmin, vmax, title, crosshair=True):
    """Display an image with pre-computed shared vmin/vmax and a colorbar."""
    if arr is None:
        ax.text(
            0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes, color="red", fontsize=9
        )
        ax.set_facecolor("#111")
        ax.set_title(title, fontsize=8)
        ax.axis("off")
        return
    im = ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if crosshair:
        draw_crosshair(ax, arr)
    ax.set_title(title, fontsize=12)
    ax.axis("off")


def _build_info_text(
    row_data, band, src_idx, n_src_band, n_dip_band, date_this, date_first_all, date_last_all
):
    """Build the multi-line string for the info text panel.

    Parameters
    ----------
    row_data       : pd.Series  — one row of the manifest DataFrame
    band           : str
    src_idx        : int        — 1-based index within the band
    n_src_band     : int        — total sources in this band
    n_dip_band     : int        — total dipoles in this band
    date_this      : str        — YYYY-MM-DD of this visit
    date_first_all : str        — date of first detection (all bands)
    date_last_all  : str        — date of last detection (all bands)
    """
    r = row_data
    visit = r.get("r:visit", "?")
    detector = r.get("r:detector", "?")
    mjd = float(r["r:midpointMjdTai"])
    snr = float(r.get("r:snr", np.nan))
    is_dip = bool(r["isDipole"])

    psf_f = float(r.get("r:psfFlux", np.nan))
    psf_e = float(r.get("r:psfFluxErr", np.nan))
    sci_f = float(r.get("r:scienceFlux", np.nan))
    sci_e = float(r.get("r:scienceFluxErr", np.nan))
    tmp_f = float(r.get("r:templateFlux", np.nan))
    tmp_e = float(r.get("r:templateFluxErr", np.nan))

    # pixel centroid  (various column names tried)
    xpix = float(r.get("r:xPix", r.get("r:x", np.nan)))
    ypix = float(r.get("r:yPix", r.get("r:y", np.nan)))
    xpix_str = f"{xpix:.1f}" if np.isfinite(xpix) else "?"
    ypix_str = f"{ypix:.1f}" if np.isfinite(ypix) else "?"

    # dipole parameters
    dip_flux_diff = float(r.get("r:dipoleFluxDiff", np.nan))
    dip_length = float(r.get("r:dipoleLength", np.nan))
    dip_angle = float(r.get("r:dipoleAngle", np.nan))

    # magnitudes
    psf_m = flux_to_mag_AB(psf_f)
    sci_m = flux_to_mag_AB(sci_f)
    tmp_m = flux_to_mag_AB(tmp_f)

    def _fm(f, e=None):
        if not np.isfinite(f):
            return "n/a"
        s = f"{f:+.1f}"
        if e is not None and np.isfinite(e):
            s += f" ±{e:.1f}"
        return s + " nJy"

    def _mag(m):
        return f"{m:.3f} mag" if np.isfinite(m) else "(neg flux)"

    lines = []
    lines.append("diaObjectId")
    lines.append(f"{DIAOBJECT_ID}")
    lines.append("")
    lines.append(f"Gaia G  : {G_str} mag")
    lines.append(f"status  : {gaia_status}")
    lines.append(f"group   : {fink_group}")
    lines.append(f"DR3     : {gaia_dr3_id}")
    lines.append("")
    lines.append(f"Visit #{src_idx}/{n_src_band} — band {band}")
    lines.append(f"visit   : {visit}")
    lines.append(f"det     : {detector}")
    lines.append(f"(x,y)   : ({xpix_str}, {ypix_str}) pix")
    lines.append(f"date    : {date_this}")
    lines.append(f"MJD     : {mjd:.4f}")
    lines.append(f"SNR     : {snr:.1f}" if np.isfinite(snr) else "SNR     : ?")
    lines.append("")
    lines.append("─── Fluxes ──────────────────")
    lines.append(f"psfFlux : {_fm(psf_f, psf_e)}")
    lines.append(f"  → {_mag(psf_m)}")
    lines.append(f"sciFlux : {_fm(sci_f, sci_e)}")
    lines.append(f"  → {_mag(sci_m)}")
    lines.append(f"tmplFlux: {_fm(tmp_f, tmp_e)}")
    lines.append(f"  → {_mag(tmp_m)}")
    lines.append("")
    dip_sym = "\U0001f534 DIPOLE" if is_dip else "\u25e6 no-dipole"
    lines.append(f"isDipole: {dip_sym}")
    if is_dip:
        if np.isfinite(dip_flux_diff):
            lines.append(f"  flxDiff: {dip_flux_diff:+.1f} nJy")
        if np.isfinite(dip_length):
            lines.append(f"  length : {dip_length:.2f} px")
        if np.isfinite(dip_angle):
            lines.append(f"  angle  : {dip_angle:.1f} deg")
    lines.append(f"Ndip/N  : {n_dip_band}/{n_src_band} this band")
    lines.append("")
    lines.append("─── Temporal coverage ───────")
    lines.append(f"first   : {date_first_all}")
    lines.append(f"last    : {date_last_all}")
    lines.append(f"span    : {TSPAN_DAYS:.1f} days")
    return "\n".join(lines)


def _plot_lightcurve_panel(ax, band, highlight_mjd, highlight_label):
    """Plot the psfFlux light curve in `band` on `ax`.

    - src detections : filled circles (band colour)
    - forced phot    : open squares (band colour)
    - current visit  : gold star + vertical dashed line
    No dipole circles drawn here — kept uncluttered for images context.
    """
    bc = BAND_COLORS.get(band, "gray")

    db = df[df["r:band"] == band].sort_values("r:midpointMjdTai")
    dt = db["r:midpointMjdTai"].values.astype(float) - T0_MJD
    psf = db["r:psfFlux"].values.astype(float)
    err = db["r:psfFluxErr"].values.astype(float)

    y_min, y_max = np.percentile(psf, [2.5, 97.5])  # 95%
    y_min = min(0.0, y_min)
    y_max = y_max * 1.3

    # All src detections
    ax.errorbar(
        dt,
        psf,
        yerr=err,
        fmt="o",
        ms=4,
        color=bc,
        ecolor=bc,
        elinewidth=0.8,
        capsize=1.5,
        alpha=0.85,
        label="src",
        zorder=3,
    )

    # Forced photometry
    if not df_fp.empty and band in df_fp["r:band"].values:
        dbf = df_fp[df_fp["r:band"] == band].sort_values("r:midpointMjdTai")
        dtf = dbf["r:midpointMjdTai"].values.astype(float) - T0_MJD
        psfF = dbf["r:psfFlux"].values.astype(float)
        errF = dbf["r:psfFluxErr"].values.astype(float)
        ax.errorbar(
            dtf,
            psfF,
            yerr=errF,
            fmt="s",
            ms=4,
            markerfacecolor="none",
            markeredgecolor=bc,
            markeredgewidth=1.1,
            ecolor=bc,
            elinewidth=0.6,
            capsize=1.5,
            alpha=0.65,
            label="fp",
            zorder=2,
        )

    # Highlight current visit with a gold star + vertical line
    dt_cur = highlight_mjd - T0_MJD
    row_cur = db[np.abs(db["r:midpointMjdTai"].values - highlight_mjd) < 1e-5]
    y_cur = float(row_cur.iloc[0]["r:psfFlux"]) if not row_cur.empty else 0.0

    ax.axvline(dt_cur, color="black", lw=1.2, ls="--", alpha=0.55, zorder=4)
    ax.scatter(
        [dt_cur],
        [y_cur],
        marker="*",
        s=220,
        color="gold",
        edgecolors="black",
        linewidths=0.8,
        zorder=6,
        label=highlight_label,
    )

    ax.axhline(0, color="k", lw=0.6, ls=":", alpha=0.4)
    ax.set_ylabel("psfFlux [nJy]", fontsize=7)
    ax.set_xlabel(f"\u0394t [days from {T0_DATE}]", fontsize=7)
    ax.tick_params(labelsize=8)
    ax.set_ylim(y_min, y_max)
    ax.legend(fontsize=8, ncol=3, loc="upper right", framealpha=0.7)
    ax.set_title(f"LC band {band}", fontsize=8, color=bc, fontweight="bold")
    ax.grid(True, alpha=0.25)


# ─── Main loop: one figure per diaSource ────────────────────────────────────
for band in bands_to_plot:
    df_band = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)
    if MAX_ROWS_PER_BAND:
        df_band = df_band.head(MAX_ROWS_PER_BAND)

    n_rows_band = len(df_band)
    n_dip_band = int(df_band["isDipole"].sum())
    bcolor = BAND_COLORS.get(band, "gray")

    print(f"\nBand {band}: {n_rows_band} diaSources  ({n_dip_band} dipoles)")

    for src_idx, (_, row) in enumerate(df_band.iterrows(), start=1):
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        mjd = float(row["r:midpointMjdTai"])
        date_this = mjd_to_date(mjd)
        is_dip = bool(row["isDipole"])

        # ── Load images ───────────────────────────────────────────────────
        sci_arr = load_npy(src_id, band, "Science")
        tmp_arr = load_npy(src_id, band, "Template")
        dif_arr = load_npy(src_id, band, "Difference")

        # Compute Sci − Template
        res_arr = None
        if sci_arr is not None and tmp_arr is not None:
            ny = min(sci_arr.shape[0], tmp_arr.shape[0])
            nx = min(sci_arr.shape[1], tmp_arr.shape[1])
            res_arr = sci_arr[:ny, :nx] - tmp_arr[:ny, :nx]

        # ── Shared colour scale — Science & Template ──────────────────────
        # Both share vmin/vmax derived from Science (or Template as fallback)
        if sci_arr is not None:
            vmin_sci, vmax_sci = zscale(sci_arr)
        elif tmp_arr is not None:
            vmin_sci, vmax_sci = zscale(tmp_arr)
        else:
            vmin_sci, vmax_sci = 0.0, 1.0

        # ── Shared symmetric scale — DIA diff & Sci−Template ─────────────
        # Derived jointly from both difference images so both share ±vmax
        ref_diff = dif_arr if dif_arr is not None else res_arr
        if ref_diff is not None and res_arr is not None:
            both = np.concatenate([ref_diff.ravel(), res_arr.ravel()])
            vmax_diff = symvlim(both)
        elif ref_diff is not None:
            vmax_diff = symvlim(ref_diff)
        else:
            vmax_diff = 1.0

        # ── Build 2×3 figure ──────────────────────────────────────────────
        fig, axes = plt.subplots(
            2,
            3,
            figsize=(15, 8),
            gridspec_kw={"hspace": 0.42, "wspace": 0.32},
        )

        ax_info = axes[0, 0]
        ax_sci = axes[0, 1]
        ax_tmp = axes[0, 2]
        ax_lc = axes[1, 0]
        ax_dif = axes[1, 1]
        ax_res = axes[1, 2]

        # ── (0,0) Info panel ──────────────────────────────────────────────
        ax_info.axis("off")
        info_txt = _build_info_text(
            row,
            band,
            src_idx,
            n_rows_band,
            n_dip_band,
            date_this,
            DATE_FIRST,
            DATE_LAST,
        )
        dip_edge = "#cc0000" if is_dip else "#555555"
        ax_info.text(
            0.04,
            0.97,
            info_txt,
            transform=ax_info.transAxes,
            fontsize=7.5,
            verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f8f8", edgecolor=dip_edge, lw=1.5),
        )

        # ── (0,1) Science ─────────────────────────────────────────────────
        sci_f = float(row.get("r:scienceFlux", np.nan))
        sci_m = flux_to_mag_AB(sci_f)
        sci_title = f"Science\n{sci_f:+.0f} nJy" + (f"  →  {sci_m:.3f} mag" if np.isfinite(sci_m) else "")
        _show_img_shared(ax_sci, sci_arr, CMAP_SCI, vmin_sci, vmax_sci, sci_title)

        # ── (0,2) Template — same scale as Science ────────────────────────
        tmp_f = float(row.get("r:templateFlux", np.nan))
        tmp_m = flux_to_mag_AB(tmp_f)
        tmp_title = f"Template  [same scale as Science]\n{tmp_f:+.0f} nJy" + (
            f"  →  {tmp_m:.3f} mag" if np.isfinite(tmp_m) else ""
        )
        _show_img_shared(ax_tmp, tmp_arr, CMAP_SCI, vmin_sci, vmax_sci, tmp_title)

        # ── (1,0) Light curve in this band ────────────────────────────────
        _plot_lightcurve_panel(ax_lc, band, mjd, f"this visit ({date_this})")

        # ── (1,1) DIA difference (downloaded) ────────────────────────────
        psf_f = float(row.get("r:psfFlux", np.nan))
        psf_m = flux_to_mag_AB(psf_f)
        psf_mag_str = f"  →  {psf_m:.3f} mag" if (np.isfinite(psf_f) and psf_f > 0) else "  (neg flux)"
        dif_title = f"DIA Diff (downloaded)\npsfFlux = {psf_f:+.0f} nJy{psf_mag_str}"
        _show_img_shared(ax_dif, dif_arr, CMAP_DIFF, -vmax_diff, vmax_diff, dif_title)

        # ── (1,2) Sci − Template (computed) ──────────────────────────────
        _show_img_shared(
            ax_res,
            res_arr,
            CMAP_DIFF,
            -vmax_diff,
            vmax_diff,
            "Sci \u2212 Template (computed)\n[same \u00b1scale as DIA Diff]",
        )

        # ── Suptitle ──────────────────────────────────────────────────────
        dip_str = "  \U0001f534 DIPOLE" if is_dip else "  \u25e6 no-dipole"
        fig.suptitle(
            f"diaObjectId={DIAOBJECT_ID}   band={band}   visit={visit}   "
            f"{date_this}{dip_str}\n"
            f"Fink: {fink_group}   Gaia G={G_str}   status={gaia_status}   "
            f"#{src_idx}/{n_rows_band}   {n_dip_band} dipoles total",
            fontsize=12,
            y=1.01,
            color=bcolor,
            fontweight="bold",
        )

        plt.tight_layout()

        if SAVE_FIGS:
            stem = f"cutouts2x3_obj{DIAOBJECT_ID}_band{band}_{src_idx:03d}_visit{visit}"
            for ext in ("pdf", "png"):
                plt.savefig(
                    os.path.join(DIR_FIGS, f"{stem}.{ext}"),
                    bbox_inches="tight",
                    dpi=120,
                )
            print(f"  \u2192 saved {stem}.{{pdf,png}}")

        plt.show()
        plt.close(fig)

## 7. Summary table

In [ ]:
from IPython.display import display as ipy_display

wanted = [
    "r:band",
    "r:visit",
    "r:detector",
    "r:midpointMjdTai",
    "r:snr",
    "r:psfFlux",
    "r:scienceFlux",
    "r:templateFlux",
    "isDipole",
    "r:dipoleFluxDiff",
    "r:dipoleLength",
    "r:dipoleAngle",
]
cols = [c for c in wanted if c in df.columns]

df_sum = df[cols].copy()
df_sum["mag_sci"] = df["r:scienceFlux"].apply(flux_to_mag_AB)
df_sum["mag_temp"] = df["r:templateFlux"].apply(flux_to_mag_AB)
df_sum["mag_psf"] = df["r:psfFlux"].apply(
    lambda f: flux_to_mag_AB(f) if (pd.notna(f) and float(f) > 0) else np.nan
)
df_sum["obs_date"] = df["r:midpointMjdTai"].apply(mjd_to_date)

print(f"diaObjectId={DIAOBJECT_ID}  |  Gaia G={G_str}  |  status={gaia_status}  |  group={fink_group}")
ipy_display(df_sum.sort_values(["r:band", "r:midpointMjdTai"]).reset_index(drop=True))

print("\nDipole fraction per band:")
for b in BAND_ORDER:
    db = df[df["r:band"] == b]
    if not len(db):
        continue
    nd = db["isDipole"].sum()
    bar = "\u2588" * int(nd / len(db) * 20)
    print(f"  {b}  {nd:3d}/{len(db):3d}  ({100 * nd / len(db):5.1f}%)  {bar}")

## 8. psfFlux light curves — 3-panel display (flux / magnitude / luptitude)

Three separate figures, each showing one row per available band:

- **Panel A — Flux (nJy)** : raw psfFlux, zero-line reference, dipole rings.
- **Panel B — AB magnitude** : log scale; non-positive flux shown as 5σ upper-limit downward arrows.
- **Panel C — Luptitude** : asinh magnitude (Lupton et al. 1999), handles all negative DIA flux.

Each subplot carries a stats box with N_src, N_fp, mean, σ (flux or mag), and dipole count.

In [ ]:
bands_lc = [b for b in BAND_ORDER if b in df["r:band"].values]


def _add_band_stats(ax, db, db_fp, band, mode):
    """Overlay a stats annotation box showing N, mean, sigma, dipole fraction."""
    psf_src = db["r:psfFlux"].values.astype(float)
    psf_src_pos = psf_src[np.isfinite(psf_src) & (psf_src > 0)]
    n_src = len(db)
    n_fp = len(db_fp) if db_fp is not None else 0
    n_dip_b = int(db["isDipole"].sum())

    if mode == "flux":
        mu = np.nanmean(psf_src) if len(psf_src) > 0 else np.nan
        sig = np.nanstd(psf_src) if len(psf_src) > 0 else np.nan
        stat_str = (
            (f"N_src={n_src}  N_fp={n_fp}  dip={n_dip_b}\n<f>={mu:+.1f}  \u03c3={sig:.1f} nJy")
            if np.isfinite(mu)
            else f"N_src={n_src}  N_fp={n_fp}"
        )

    elif mode in ("mag", "luptitude"):
        if len(psf_src_pos) >= 2:
            mags = -2.5 * np.log10(psf_src_pos / AB_FLUX_ZERO)
            mu_m = np.nanmean(mags)
            sig_m = np.nanstd(mags)
            stat_str = f"N_src={n_src}  N_fp={n_fp}  dip={n_dip_b}\n<mag>={mu_m:.3f}  \u03c3={sig_m:.4f}"
        else:
            stat_str = f"N_src={n_src}  N_fp={n_fp}  dip={n_dip_b}"
    else:
        stat_str = f"N_src={n_src}  N_fp={n_fp}"

    ax.text(
        0.01,
        0.01,
        stat_str,
        transform=ax.transAxes,
        fontsize=7,
        va="bottom",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.85),
    )


for mode in ("flux", "mag", "luptitude"):
    fig, axes = plt.subplots(
        len(bands_lc),
        1,
        figsize=(11, 2.8 * len(bands_lc)),
        sharex=True,
        squeeze=False,
    )
    ylabel_map = {"flux": "psfFlux [nJy]", "mag": "AB mag", "luptitude": "asinh mag"}

    for ax_i, band in enumerate(bands_lc):
        ax = axes[ax_i][0]
        bc = BAND_COLORS.get(band, "gray")

        db = df[df["r:band"] == band].sort_values("r:midpointMjdTai")
        dt = db["r:midpointMjdTai"].values.astype(float) - T0_MJD
        psf = db["r:psfFlux"].values.astype(float)
        err = db["r:psfFluxErr"].values.astype(float)
        dip = db["isDipole"].values.astype(bool)

        db_fp_b = None
        dtf = psf_fp = err_fp = np.array([])
        if not df_fp.empty and band in df_fp["r:band"].values:
            db_fp_b = df_fp[df_fp["r:band"] == band].sort_values("r:midpointMjdTai")
            dtf = db_fp_b["r:midpointMjdTai"].values.astype(float) - T0_MJD
            psf_fp = db_fp_b["r:psfFlux"].values.astype(float)
            err_fp = db_fp_b["r:psfFluxErr"].values.astype(float)

        # ── Convert y values per mode ─────────────────────────────────────
        if mode == "flux":
            y_src, ye_src = psf, err
            y_fp, ye_fp = psf_fp, err_fp
            y_min, y_max = np.percentile(y_src, [2.5, 97.5])  # 95%
            y_min = min(0.0, y_min)
            y_max = 1.2 * y_max
            ax.axhline(0, color="k", lw=0.6, ls=":", alpha=0.4)

        elif mode == "mag":
            with np.errstate(invalid="ignore", divide="ignore"):
                m_src = np.where(psf > 0, -2.5 * np.log10(psf / AB_FLUX_ZERO), np.nan)
                me_src = np.where(
                    (psf > 0) & np.isfinite(err),
                    2.5 / np.log(10) * np.abs(err / psf),
                    np.nan,
                )
                m_fp = np.where(psf_fp > 0, -2.5 * np.log10(psf_fp / AB_FLUX_ZERO), np.nan)
                me_fp = np.where(
                    (psf_fp > 0) & np.isfinite(err_fp),
                    2.5 / np.log(10) * np.abs(err_fp / psf_fp),
                    np.nan,
                )
            y_src, ye_src = m_src, me_src
            y_min, y_max = np.percentile(y_src, [2.5, 97.5])  # 95%
            y_fp, ye_fp = m_fp, me_fp

            # Upper-limit arrows for non-positive flux
            neg_mask = np.isfinite(psf) & (psf <= 0)
            if neg_mask.any():
                ul_flux = 5.0 * err[neg_mask]
                ul_mag = np.where(ul_flux > 0, -2.5 * np.log10(ul_flux / AB_FLUX_ZERO), np.nan)
                valid_ul = np.isfinite(ul_mag)
                if valid_ul.any():
                    ax.scatter(
                        dt[neg_mask][valid_ul],
                        ul_mag[valid_ul],
                        marker="v",
                        s=40,
                        color=bc,
                        alpha=0.5,
                        label="5\u03c3 UL",
                        zorder=2,
                    )
            ax.invert_yaxis()

        elif mode == "luptitude":
            all_err = np.concatenate([err, err_fp]) if len(err_fp) > 0 else err
            valid_err = all_err[np.isfinite(all_err) & (all_err > 0)]
            b_soft = float(np.nanmedian(valid_err)) if len(valid_err) > 0 else 1.0
            y_src, _ = flux_to_luptitude(psf, b_soft)
            ye_src = 1.085736 * err / np.sqrt(psf**2 + (2 * b_soft) ** 2)
            y_min, y_max = np.percentile(y_src, [2.5, 97.5])  # 95%
            if len(psf_fp) > 0:
                y_fp, _ = flux_to_luptitude(psf_fp, b_soft)
                ye_fp = 1.085736 * err_fp / np.sqrt(psf_fp**2 + (2 * b_soft) ** 2)
            ax.invert_yaxis()

        # ── Dipole overlay ────────────────────────────────────────────────
        if dip.any():
            ax.scatter(
                dt[dip],
                y_src[dip],
                marker="o",
                s=110,
                facecolors="none",
                edgecolors="lightgrey",
                linewidths=1.5,
                zorder=5,
                alpha=0.65,
                label="dipole",
            )

        # ── Errorbars ─────────────────────────────────────────────────────
        valid_s = np.isfinite(y_src) & np.isfinite(ye_src)
        if valid_s.any():
            ax.errorbar(
                dt[valid_s],
                y_src[valid_s],
                yerr=ye_src[valid_s],
                fmt="o",
                ms=4,
                color=bc,
                ecolor=bc,
                elinewidth=0.9,
                capsize=2,
                alpha=0.9,
                label="src",
                zorder=3,
            )

        if len(dtf) > 0:
            valid_f = np.isfinite(y_fp) & np.isfinite(ye_fp)
            if valid_f.any():
                ax.errorbar(
                    dtf[valid_f],
                    y_fp[valid_f],
                    yerr=ye_fp[valid_f],
                    fmt="s",
                    ms=4,
                    markerfacecolor="none",
                    markeredgecolor=bc,
                    markeredgewidth=1.1,
                    ecolor=bc,
                    elinewidth=0.7,
                    capsize=2,
                    alpha=0.65,
                    label="fp",
                    zorder=2,
                )

        ax.set_ylabel(f"{ylabel_map[mode]}\nband {band}", fontsize=8)
        if mode == "flux":
            ax.set_ylim(y_min, y_max)
        ax.tick_params(labelsize=10)
        ax.legend(
            fontsize=12,
            ncol=1,
            bbox_to_anchor=(1.01, 0.5),
            loc="center left",
            frameon=False,
        )
        ax.grid(True, alpha=0.25)
        _add_band_stats(ax, db, db_fp_b, band, mode)

    axes[-1][0].set_xlabel(f"\u0394t [days from MJD {T0_MJD:.2f}  ({T0_DATE})]", fontsize=9)

    mode_titles = {
        "flux": "psfFlux [nJy]  —  src (\u25cf) + fp (\u25ab) + dipoles (\u25cb)",
        "mag": "AB magnitude  —  src (\u25cf) + fp (\u25ab) + 5\u03c3 UL (\u25bd) + dipoles (\u25cb)",
        "luptitude": "Luptitude (asinh mag)  —  src (\u25cf) + fp (\u25ab) + dipoles (\u25cb)",
    }
    fig.suptitle(
        f"diaObjectId={DIAOBJECT_ID}  |  {mode_titles[mode]}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}  "
        f"first={DATE_FIRST}  last={DATE_LAST}",
        fontsize=10,
        y=1.0,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"lc_{mode}_obj{DIAOBJECT_ID}"
        for ext in ("pdf", "png"):
            plt.savefig(
                os.path.join(DIR_FIGS, f"{stem}.{ext}"),
                bbox_inches="tight",
                dpi=120,
            )
        print(f"\u2192 saved {stem}.{{pdf,png}}")

    plt.show()